# Batch Dynamic Extraction: All Hackathon Project Bundles

This notebook uses the reusable `extraction_pipeline` Python package instead of keeping the pipeline logic in notebook cells.

It scans the whole `Hackathon/` folder, groups files into project bundles, and tries to extract the same SQL-ready fields as the `042` prototype for every bundle:

- `project_number`
- `project_name`
- `organization`
- `runtime_start`
- `runtime_end`
- `region`
- `main_indicator_name`
- `main_indicator_soll`
- `main_indicator_ist`
- `main_indicator_fulfillment_percent`
- `target_group_text`
- `measures_text`

The extraction is best-effort. Some future-period bundles, for example `2026_2027`, do not yet have `Indikatorenbericht` or `Inhaltlicher Endbericht` files, so they will show missing-Ist errors rather than breaking the run.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from extraction_pipeline import (
    discover_project_bundles,
    extract_bundles,
    load_models,
    write_results,
)

DATA_ROOT = ROOT / "Hackathon"
OUTPUT_DIR = ROOT / "notebooks" / "outputs" / "batch_dynamic"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 250)

/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Discover project bundles

In [2]:
bundles = discover_project_bundles(DATA_ROOT)
bundle_df = pd.DataFrame([bundle.as_dict() for bundle in bundles])
print(f"Discovered {len(bundles)} project bundles")
bundle_df

Discovered 14 project bundles


,key,year_folder,project_number,project_slug,soll_excel,ist_excel,project_description,final_report,file_count
0,2019_2020:35,2019_2020,35,deutsch-im-alltag,/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/Hackathon/2019_2020/035_DEUTSCH-IM-ALLTAG_2019_2020_Indikatoren.xlsx,/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/Hackathon/2019_2020/035_DEUTSCH-IM-ALLTAG_2019_2020_Indikatorenbericht.xlsx,/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/Hackathon/2019_2020/035_DEUTSCH-IM-ALLTAG_2019_2020_Projektbeschreibung.docx,/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/Hackathon/2019_2020/035_DEUTSCH-IM-ALLTAG_2019_2020_Inhaltlicher Endbericht.docx,4
1,2019_2020:93,2019_2020,93,youthconnect,/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/Hackathon/2019_2020/093_YouthConnect_2019_2020_Indikatoren.xlsx,/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/Hackathon/2019_2020/093_YouthConnect_2019_2020_Indikatorenbericht.xlsx,/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/Hackathon/2019_2020/093_YouthConnect_2019_2020_Projektbeschreibung.docx,/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/Hackathon/2019_2020/093_YouthConnect_2019_2020 Inhaltlicher Endbericht.docx,4
2,2019_2020:165,2019_2020,165,begegnungsraeume,/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/Hackathon/2019_2020/165_Begegnungsraeume_2019_2020_Indikatoren.xlsx,/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/Hackathon/2019_2020/165_Begegnungsraeume_2019_2020_Indikatorenbericht.xls,/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/Hackathon/2019_2020/165_Begegnungsraeume_2019_2020_Projektbeschreibung.docx,/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/Hackathon/2019_2020/165_Begegnungsraeume_2019_2020_Inhaltlicher Endbericht.docx,4
3,2021:15,2021,15,schulstart,/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/Hackathon/2021/015_SchulStart_2021_Indikatoren.xlsx,/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/Hackathon/2021/015_SchulStart_2021_Indikatorenbericht.xlsx,/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/Hackathon/2021/015_SchulStart_2021_Projektbeschreibung.pdf,/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/Hackathon/2021/015_SchulStart_2021_Inhaltlicher Endbericht.docx,4
4,2021:57,2021,57,handwerk-zukunft,/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/Hackathon/2021/057_HandWerk.Zukunft_2021_Indikatoren.xlsx,/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/Hackathon/2021/057_HandWerk.Zukunft_2021_Indikatorenbericht.xlsx,/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/Hackathon/2021/57_HandWerk.Zukunft_2021_Projektbeschreibung.pdf,/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/Hackathon/2021/057_HandWerk.Zukunft_2021_Inhaltlicher Endbericht.docx,4
5,2022_2023:50,2022_2023,50,lernstube,/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/Hackathon/2022_2023/050_Lernstube_2022_2023_Indikatoren.xlsx,/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/Hackathon/2022_2023/050_Lernstube_2022_2023_Indikatorenbericht.xlsx,/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/Hackathon/2022_2023/050_Lernstube_2022_2023_Projektbeschreibung.pdf,/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/Hackathon/2022_2023/050_Lernstube_2022_2023_Inhaltlicher Endbericht.docx,4
6,2022_2023:96,2022_2023,96,wurzeln-fassen,/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/Hackathon/2022_2023/096_Wurzeln-fassen_2022_2023_Indikatoren.xlsx,/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/Hackathon/2022_2023/096_Wurzeln-fassen_2022_2023_Indikatorenbericht.xlsx,/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/Hackathon/2022_2023/096_Wurzeln-fassen_2022_2023_Projektbeschreibung.pdf,/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/Hackathon/2022_2023/096_Wurzeln-fassen_2022_2023_Inhaltlicher Endbericht.docx,4
7,2022_2023:130,2022_2023,130,handwerk-zukunft,/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/Hackathon/2022_2023/

## 2. Load transformer models

This loads:

- `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` for German semantic matching in Excel and Word evidence selection.
- `urchade/gliner_multi-v2.1` for German NER from Word/PDF text.

If you want a faster smoke test, set `LOAD_GLINER = False` and/or `LIMIT = 2` in the next cell.

In [3]:
LOAD_GLINER = True
LIMIT = None  # Set to e.g. 2 for a quick smoke test. None means all discovered bundles.

models = load_models(load_gliner=LOAD_GLINER)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 384.88it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/.venv/lib64/python3.14/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 29046.43it/s]


## 3. Run extraction for all bundles

In [4]:
results_df, extraction_results = extract_bundles(bundles, models, limit=LIMIT)
results_df

/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/.venv/lib64/python3.14/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 407 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/.venv/lib64/python3.14/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 2960 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/.venv/lib64/python3.14/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 484 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/.venv/lib64/python3.14/site-packages/gliner/data_processing/pr

,project_key,year_folder,project_number_from_filename,project_number,project_name,organization,runtime_start,runtime_end,region,main_indicator_name,main_indicator_soll,main_indicator_ist,main_indicator_fulfillment_percent,target_group_text,measures_text,has_soll_excel,has_ist_excel,has_project_description,has_final_report,error_count,errors
0,2019_2020:35,2019_2020,35,35,Deutsch im Alltag – Schritt für Schritt,Verein SprachBrücke Wien,Verein SprachBrücke Wien,#DIV/0!,Verein SprachBrücke Wien,Anzahl der Teilnehmerinnen und Teilnehmer im Projekt,80.0,76.0,95.0,"DEUTSCH IM ALLTAG ist ein international erfolgreiches, wissensbasiertes Programm zur frühen Bildungsförderung in Familien mit Migrationserfahrung. Es zielt darauf ab, die Entwicklungsmöglichkeiten von Kindern frühzeitig und nachhaltig zu verbesse...",Erfahrungen in der Projektabwicklung,True,True,True,True,0,
1,2019_2020:93,2019_2020,93,093-2019,YouthConnect – Jugendliche gestalten Integration,Jugendzentrum Bunterhaufen,Jugendzentrum Bunterhaufen,#DIV/0!,Jugendzentrum Bunterhaufen,Anzahl der Teilnehmerinnen und Teilnehmer im Projekt,470.0,731.0,155.5,"YOUTHCONNECT bietet niederschwellige Sport- und Freizeitangebote für Kinder und Jugendliche (6-21 Jahren) der NAP.I.-Zielgruppen, insbesondere Konventionsflüchtlinge, Subsidiär Schutzberechtigte sowie mehrfach benachteiligte MigrantInnen mit beso...",Erfahrungen in der Projektabwicklung,True,True,True,True,0,
2,2019_2020:165,2019_2020,165,165-2019&2020,Banonda - Dialog und Integration,Diakonie Flüchtlingsdienst gem. GmbH,Diakonie Flüchtlingsdienst gem. GmbH,7,Diakonie Flüchtlingsdienst gem. GmbH,Anzahl der Teilnehmerinnen und Teilnehmer im Projekt,510.0,509.0,99.8,BACH Stütz- und Förderangebote für Jugendliche und junge Erwachsene mit Migrationsbiographie und erhöhtem Förderbedarf (NÖ); Laufzeit: 01.01.2018 – 31.12.2018,Erfahrungen in der Projektabwicklung,True,True,True,True,0,
3,2021:15,2021,15,03-15-2021,die chance,3950,3950,die chance,03-15-2021,Anzahl der Projektteilnehmerinnen und -teilnehmer gesamt,3950.0,3950.0,102.4,Migrantische Communities konnten kaum besucht werden,Abweichungen im Projektverlauf und ergriffene Maßnahmen,True,True,True,True,0,
4,2021:57,2021,57,57-2021,HandWerk.Zukunft – Berufsvorbereitungskurse mit Lehrvermittlung,135 beratene Personen (45 Kurs-TN),135 beratene Personen (45 Kurs-TN),HandWerk.Zukunft – Berufsvorbereitungskurse mit Lehrvermittlung,57-2021,Anzahl der Projektteilnehmerinnen und -teilnehmer gesamt,135.0,135.0,103.7,Ziel 2: Förderung der beruflichen Qualifizierung sowie der ausbildungsadäquaten Beschäftigung von Menschen mit Migrationshintergrund,Ziel 4: Information und Beratung zum Lehrstellenmarkt sowie Berufsorientierung und Mentoring,True,True,True,True,0,
5,2022_2023:50,2022_2023,50,Niederösterreich,Lernstube - Kostenlose Lernbegleitung für Kinder und Jugendliche,,2022-01-01,2023-12-31,Tirol,Anzahl der Projektteilnehmenden gesamt,941.0,941.0,124.0,"Die Projektzielgruppe, nämlich Kinder und Jugendliche von 6 bis 15 Jahren, die Drittstaatsangehörigen mit längerfristiger Aufenthaltsperspektive, Asylberechtigte und subsidiär Schutzberechtigte, Vertriebene gemäß § 62 AsylG 2005, EU-Bürger*innen,...","Bitte geben Sie hier nur Ihre Kurzbeschreibung ein (max. 1500 Zeichen)! Diese sollte die durchgeführten Maßnahmen, die erreichte Wirkung sowie wesentliche Abweichungen vom ursprünglichen Projektplan (Projektbeschreibung) enthalten. Die Indikatore...",True,True,True,True,0,
6,2022_2023:96,2022_2023,96,Niederösterreich,Wurzeln fassen – Traumasensible Begleitung,,2022-01-01,2023-12-31,Wien,Anzahl der Projektteilnehmenden gesamt,818.0,818.0,147.2,Abweichungen vom Projektverlauf hinsichtlich der Zielgruppe der Vertriebenen gemäß § 62 AsylG 2005,"ANNEX II. Kurzübersicht zur Projektumsetzung Zum Beispiel: übergeordnetes Ziel: Arbeitsmarktintegration von Jugendlichen; Handlungsziel: Vermittlung einer Lehr- Geben Sie eine kurze Übersicht zur Projektumsetzung und achten Sie darauf, stelle; Be...",Tr

## 4. Save batch outputs

In [5]:
csv_path = OUTPUT_DIR / "batch_sql_ready_results.csv"
xlsx_path = OUTPUT_DIR / "batch_sql_ready_results.xlsx"

results_df.to_csv(csv_path, index=False)
results_df.to_excel(xlsx_path, index=False)
write_results(extraction_results, OUTPUT_DIR / "evidence")

print("Wrote:")
print(csv_path.relative_to(ROOT))
print(xlsx_path.relative_to(ROOT))
print((OUTPUT_DIR / "evidence").relative_to(ROOT))

Wrote:
notebooks/outputs/batch_dynamic/batch_sql_ready_results.csv
notebooks/outputs/batch_dynamic/batch_sql_ready_results.xlsx
notebooks/outputs/batch_dynamic/evidence


## 5. Review extraction health

In [6]:
health_cols = [
    "project_key",
    "has_soll_excel",
    "has_ist_excel",
    "has_project_description",
    "has_final_report",
    "error_count",
    "errors",
]
results_df[health_cols].sort_values(["error_count", "project_key"], ascending=[False, True])

,project_key,has_soll_excel,has_ist_excel,has_project_description,has_final_report,error_count,errors
11,2026_2027:20,True,False,True,False,1,No Indikatorenbericht/Ist Excel file available.
12,2026_2027:50,True,False,True,False,1,No Indikatorenbericht/Ist Excel file available.
13,2026_2027:60,True,False,True,False,1,No Indikatorenbericht/Ist Excel file available.
2,2019_2020:165,True,True,True,True,0,
0,2019_2020:35,True,True,True,True,0,
1,2019_2020:93,True,True,True,True,0,
3,2021:15,True,True,True,True,0,
4,2021:57,True,True,True,True,0,
7,2022_2023:130,True,True,True,True,0,
5,2022_2023:50,True,True,True,True,0,


## 6. Compact SQL-ready view

In [7]:
sql_cols = [
    "project_key",
    "project_number",
    "project_name",
    "organization",
    "runtime_start",
    "runtime_end",
    "region",
    "main_indicator_name",
    "main_indicator_soll",
    "main_indicator_ist",
    "main_indicator_fulfillment_percent",
    "target_group_text",
    "measures_text",
]
results_df[sql_cols]

,project_key,project_number,project_name,organization,runtime_start,runtime_end,region,main_indicator_name,main_indicator_soll,main_indicator_ist,main_indicator_fulfillment_percent,target_group_text,measures_text
0,2019_2020:35,35,Deutsch im Alltag – Schritt für Schritt,Verein SprachBrücke Wien,Verein SprachBrücke Wien,#DIV/0!,Verein SprachBrücke Wien,Anzahl der Teilnehmerinnen und Teilnehmer im Projekt,80.0,76.0,95.0,"DEUTSCH IM ALLTAG ist ein international erfolgreiches, wissensbasiertes Programm zur frühen Bildungsförderung in Familien mit Migrationserfahrung. Es zielt darauf ab, die Entwicklungsmöglichkeiten von Kindern frühzeitig und nachhaltig zu verbesse...",Erfahrungen in der Projektabwicklung
1,2019_2020:93,093-2019,YouthConnect – Jugendliche gestalten Integration,Jugendzentrum Bunterhaufen,Jugendzentrum Bunterhaufen,#DIV/0!,Jugendzentrum Bunterhaufen,Anzahl der Teilnehmerinnen und Teilnehmer im Projekt,470.0,731.0,155.5,"YOUTHCONNECT bietet niederschwellige Sport- und Freizeitangebote für Kinder und Jugendliche (6-21 Jahren) der NAP.I.-Zielgruppen, insbesondere Konventionsflüchtlinge, Subsidiär Schutzberechtigte sowie mehrfach benachteiligte MigrantInnen mit beso...",Erfahrungen in der Projektabwicklung
2,2019_2020:165,165-2019&2020,Banonda - Dialog und Integration,Diakonie Flüchtlingsdienst gem. GmbH,Diakonie Flüchtlingsdienst gem. GmbH,7,Diakonie Flüchtlingsdienst gem. GmbH,Anzahl der Teilnehmerinnen und Teilnehmer im Projekt,510.0,509.0,99.8,BACH Stütz- und Förderangebote für Jugendliche und junge Erwachsene mit Migrationsbiographie und erhöhtem Förderbedarf (NÖ); Laufzeit: 01.01.2018 – 31.12.2018,Erfahrungen in der Projektabwicklung
3,2021:15,03-15-2021,die chance,3950,3950,die chance,03-15-2021,Anzahl der Projektteilnehmerinnen und -teilnehmer gesamt,3950.0,3950.0,102.4,Migrantische Communities konnten kaum besucht werden,Abweichungen im Projektverlauf und ergriffene Maßnahmen
4,2021:57,57-2021,HandWerk.Zukunft – Berufsvorbereitungskurse mit Lehrvermittlung,135 beratene Personen (45 Kurs-TN),135 beratene Personen (45 Kurs-TN),HandWerk.Zukunft – Berufsvorbereitungskurse mit Lehrvermittlung,57-2021,Anzahl der Projektteilnehmerinnen und -teilnehmer gesamt,135.0,135.0,103.7,Ziel 2: Förderung der beruflichen Qualifizierung sowie der ausbildungsadäquaten Beschäftigung von Menschen mit Migrationshintergrund,Ziel 4: Information und Beratung zum Lehrstellenmarkt sowie Berufsorientierung und Mentoring
5,2022_2023:50,Niederösterreich,Lernstube - Kostenlose Lernbegleitung für Kinder und Jugendliche,,2022-01-01,2023-12-31,Tirol,Anzahl der Projektteilnehmenden gesamt,941.0,941.0,124.0,"Die Projektzielgruppe, nämlich Kinder und Jugendliche von 6 bis 15 Jahren, die Drittstaatsangehörigen mit längerfristiger Aufenthaltsperspektive, Asylberechtigte und subsidiär Schutzberechtigte, Vertriebene gemäß § 62 AsylG 2005, EU-Bürger*innen,...","Bitte geben Sie hier nur Ihre Kurzbeschreibung ein (max. 1500 Zeichen)! Diese sollte die durchgeführten Maßnahmen, die erreichte Wirkung sowie wesentliche Abweichungen vom ursprünglichen Projektplan (Projektbeschreibung) enthalten. Die Indikatore..."
6,2022_2023:96,Niederösterreich,Wurzeln fassen – Traumasensible Begleitung,,2022-01-01,2023-12-31,Wien,Anzahl der Projektteilnehmenden gesamt,818.0,818.0,147.2,Abweichungen vom Projektverlauf hinsichtlich der Zielgruppe der Vertriebenen gemäß § 62 AsylG 2005,"ANNEX II. Kurzübersicht zur Projektumsetzung Zum Beispiel: übergeordnetes Ziel: Arbeitsmarktintegration von Jugendlichen; Handlungsziel: Vermittlung einer Lehr- Geben Sie eine kurze Übersicht zur Projektumsetzung und achten Sie darauf, stelle; Be..."
7,2022_2023:130,05-130-2022/23,HandWerk.Zukunft – Berufsvorbereitungskurse mit Lehrvermittlung,,2022-01-01,2023-12-31,Steiermark,Anzahl der Projektteilnehmenden gesamt,180.0,180.0,113.9,Die vertraglich vereinbarten Zielgruppen konnten sehr gut erreicht werden. Dabei handelte es sich um Jugendliche und junge Erwachsene zwischen 15 und 25 Jahren mit Mi

## 7. Read one row in full

In [8]:
ROW_INDEX = 0

def format_result_row(row):
    lines = [f"# Extraction result: {row['project_key']}", ""]
    for field, value in row.items():
        if pd.isna(value):
            value = ""
        lines.append(f"## `{field}`")
        lines.append(str(value).strip() if str(value).strip() else "_empty_")
        lines.append("")
    return "\n".join(lines)

display(Markdown(format_result_row(results_df.iloc[ROW_INDEX])))

# Extraction result: 2019_2020:35

## `project_key`
2019_2020:35

## `year_folder`
2019_2020

## `project_number_from_filename`
35

## `project_number`
35

## `project_name`
Deutsch im Alltag – Schritt für Schritt

## `organization`
Verein SprachBrücke Wien

## `runtime_start`
Verein SprachBrücke Wien

## `runtime_end`
#DIV/0!

## `region`
Verein SprachBrücke Wien

## `main_indicator_name`
Anzahl der Teilnehmerinnen und Teilnehmer im Projekt

## `main_indicator_soll`
80.0

## `main_indicator_ist`
76.0

## `main_indicator_fulfillment_percent`
95.0

## `target_group_text`
DEUTSCH IM ALLTAG ist ein international erfolgreiches, wissensbasiertes Programm zur frühen Bildungsförderung in Familien mit Migrationserfahrung. Es zielt darauf ab, die Entwicklungsmöglichkeiten von Kindern frühzeitig und nachhaltig zu verbessern. Zielgruppe sind sozial- und bildungsbenachteiligte Familien mit Kindern im Alter von 3 bis 7 Jahren. In aufsuchender Familienarbeit durch muttersprachliche Hausbesucherinnen werden innerfamiliäre Bildungsaktivitäten auf spielerische Weise angeregt und vertieft und die Kinder auf den Schulbesuch vorbereitet. Durch begleitende Gruppentreffen und Exkursionen im sozialen Umfeld werden Kultur, Werte und Lebensweisen der Aufnahmegesellschaft vermittelt. Die Bildungssensibilisierung der gesamten Familie und die Verbesserung der sozialen Integration stellen einen wichtigen Beitrag zur Chancengerechtigkeit für Familien mit Migrationserfahrung dar.

## `measures_text`
Erfahrungen in der Projektabwicklung

## `has_soll_excel`
True

## `has_ist_excel`
True

## `has_project_description`
True

## `has_final_report`
True

## `error_count`
0

## `errors`
_empty_


## 8. Save frontend records

This cell uses the already computed `results_df` and writes a frontend-friendly JSON file. It does not rerun extraction.


In [9]:
import json

frontend_cols = [
    "project_key",
    "year_folder",
    "project_number_from_filename",
    "project_number",
    "project_name",
    "organization",
    "runtime_start",
    "runtime_end",
    "region",
    "main_indicator_name",
    "main_indicator_soll",
    "main_indicator_ist",
    "main_indicator_fulfillment_percent",
    "target_group_text",
    "measures_text",
    "has_soll_excel",
    "has_ist_excel",
    "has_project_description",
    "has_final_report",
    "error_count",
    "errors",
]

frontend_records = (
    results_df[frontend_cols]
    .where(pd.notna(results_df[frontend_cols]), None)
    .to_dict(orient="records")
)

final_records_path = OUTPUT_DIR / "final_records_for_frontend.json"
final_records_path.write_text(
    json.dumps(frontend_records, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

print(f"Wrote {len(frontend_records)} records to:")
print(final_records_path.relative_to(ROOT))


Wrote 14 records to:
notebooks/outputs/batch_dynamic/final_records_for_frontend.json
